# LeetCode 2965 — Find Missing and Repeated Values bằng Python

Link đề: https://leetcode.com/problems/find-missing-and-repeated-values/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Math, Matrix

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `n x n` chứa các số từ `1` đến `n²`, **đúng một số `a` xuất hiện hai lần** và **đúng một số `b` bị thiếu**; trả về `[a, b]` (số lặp **trước**, số thiếu **sau**).
2. Biết **trải phẳng (flatten)** lưới hai chiều thành danh sách một chiều và **đếm tần suất** bằng mảng đếm.
3. Biết **công thức tổng `1 + 2 + ... + N = N(N + 1) / 2`** và tổng bình phương `N(N + 1)(2N + 1) / 6` để dựng hệ phương trình tìm `a` và `b`.
4. Đi từ cách **mảng đếm tần suất** → **`set` + hiệu tổng** → **hai phương trình (tổng và tổng bình phương)** → **XOR chia hai nhóm theo bit thấp nhất**.
5. Tránh các lỗi kinh điển: trả về `[b, a]` (sai thứ tự), dùng `n` thay vì `n * n` làm cận trên, tính `b = tổng_kỳ_vọng - tổng_thực` mà **quên bù số lặp** `a`, và `return` quá sớm khi mới thấy số lặp.

## 2. Đề bài & ví dụ

Cho lưới số nguyên `grid` kích thước `n x n` (đánh chỉ số từ `0`) chứa các giá trị trong khoảng `[1, n²]`. **Mỗi số** trong `[1, n²]` xuất hiện **đúng một lần**, **trừ** số `a` xuất hiện **hai lần** và số `b` **không xuất hiện**.

Trả về mảng `ans` gồm hai phần tử: `ans[0] = a` (số **lặp**) và `ans[1] = b` (số **thiếu**).

**Ví dụ 1**

- Input: `grid = [[1, 3], [2, 2]]`
- Output: `[2, 4]`
- Giải thích: số `2` xuất hiện hai lần, số `4` không xuất hiện (các số cần có là `1, 2, 3, 4`).

**Ví dụ 2**

- Input: `grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]`
- Output: `[9, 5]`
- Giải thích: số `9` xuất hiện hai lần, số `5` không xuất hiện (các số cần có là `1..9`).

**Ràng buộc:** `2 <= n == grid.length == grid[i].length <= 50`; `1 <= grid[i][j] <= n * n`; với mọi `x` trong `[1, n * n]` có đúng một `x` xuất hiện hai lần, đúng một `x` vắng mặt, và mọi `x` còn lại xuất hiện đúng một lần.

In [1]:
from typing import List
from collections import Counter
from itertools import chain
import random

examples = [
    ([[1, 3], [2, 2]], [2, 4]),
    ([[9, 1, 7], [8, 9, 2], [3, 4, 6]], [9, 5]),
    ([[1, 1], [3, 4]], [1, 2]),
]
for grid, expected in examples:
    print("grid =", grid, "-> kỳ vọng", expected)

grid = [[1, 3], [2, 2]] -> kỳ vọng [2, 4]
grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> kỳ vọng [9, 5]
grid = [[1, 1], [3, 4]] -> kỳ vọng [1, 2]


## 3. Kiến thức nền

### 3.1. Lưới hai chiều và số ô `n * n`

`grid` là danh sách của `n` hàng, mỗi hàng `n` số. Tổng số ô là `n * n`; ví dụ `n = 3` có `9` ô và các số cần có là `1..9`. Chú ý: cận trên là **`n * n`**, không phải `n`.

In [2]:
grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]
n = len(grid)
print(n, n * n)                          # 3 9
print(grid[0], grid[0][2])               # [9, 1, 7] 7

3 9
[9, 1, 7] 7


### 3.2. Trải phẳng (flatten) lưới thành một danh sách

Có ba cách phổ biến cho cùng kết quả:

- list comprehension hai tầng: `[v for row in grid for v in row]` (đọc **từ trái sang phải**: hàng trước, giá trị sau);
- `list(chain.from_iterable(grid))` của `itertools`;
- `sum(grid, [])` (ngắn nhưng chậm, ít dùng).

In [3]:
grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]
flat1 = [v for row in grid for v in row]
flat2 = list(chain.from_iterable(grid))
print(flat1)
print(flat1 == flat2)                    # True

[9, 1, 7, 8, 9, 2, 3, 4, 6]
True


### 3.3. Mảng đếm tần suất

Nếu giá trị nằm trong `1..N`, ta tạo `cnt = [0] * (N + 1)` rồi mỗi lần gặp `v` làm `cnt[v] += 1`. Sau đó `cnt[v] == 2` là số lặp, `cnt[v] == 0` là số thiếu. Chỉ số `0` thừa (để `cnt[v]` khớp trực tiếp giá trị `v`, tiện đọc).

In [4]:
flat = [9, 1, 7, 8, 9, 2, 3, 4, 6]
N = 9
cnt = [0] * (N + 1)
for v in flat:
    cnt[v] += 1
print(cnt)                                                    # chỉ số 0 không dùng
print("lặp:", [v for v in range(1, N + 1) if cnt[v] == 2])    # [9]
print("thiếu:", [v for v in range(1, N + 1) if cnt[v] == 0])  # [5]

[0, 1, 1, 1, 1, 0, 1, 1, 1, 2]
lặp: [9]
thiếu: [5]


### 3.4. Công thức tổng và tổng bình phương

- `1 + 2 + ... + N = N(N + 1) / 2`;
- `1² + 2² + ... + N² = N(N + 1)(2N + 1) / 6`.

Trong Python dùng `//` (chia lấy phần nguyên) vì các tử số luôn chia hết, để giữ kiểu `int` thay vì `float`.

In [5]:
N = 9
print(sum(range(1, N + 1)), N * (N + 1) // 2)                              # 45 45
print(sum(v * v for v in range(1, N + 1)), N * (N + 1) * (2 * N + 1) // 6) # 285 285

45 45
285 285


### 3.5. Tập hợp `set` và phát hiện trùng

`seen = set()`; khi gặp `v`: nếu `v in seen` thì `v` là số **lặp**, ngược lại `seen.add(v)`. Mỗi thao tác trung bình `O(1)`.

In [6]:
seen = set()
for v in [9, 1, 7, 8, 9, 2, 3, 4, 6]:
    if v in seen:
        print("gặp lại", v)
    seen.add(v)
print(sorted(seen))

gặp lại 9
[1, 2, 3, 4, 6, 7, 8, 9]


### 3.6. XOR (`^`) và tính chất triệt tiêu

`x ^ x = 0` và `x ^ 0 = x`; XOR có tính giao hoán, kết hợp. Vì vậy XOR một nhóm số mà **mỗi giá trị xuất hiện chẵn lần** cho `0`. Ta sẽ dùng để "triệt tiêu" các số xuất hiện đúng 2 lần. `x & -x` giữ lại **bit `1` thấp nhất** của `x`.

In [7]:
print(5 ^ 5, 5 ^ 0, 3 ^ 5 ^ 3)           # 0 5 5
print(6 ^ 5, bin(6 ^ 5))                 # 3 0b11
x = 12                                    # 0b1100
print(bin(x & -x), x & -x)                # 0b100 4: bit 1 thấp nhất

0 5 5
3 0b11
0b100 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung có `n²` **hộp đánh số** `1..N` (với `N = n²`). Mỗi giá trị trong lưới là một lần "bỏ bi vào hộp". Hộp đúng phải có đúng một bi; kiểm tra xem hộp nào có **hai** bi và hộp nào **trống**. Với `[[1, 3], [2, 2]]` (`N = 4`):

| hộp `v` | số bi (số lần xuất hiện) | nhận xét |
|---|---|---|
| 1 | 1 | bình thường |
| 2 | **2** | **lặp** `a = 2` |
| 3 | 1 | bình thường |
| 4 | **0** | **thiếu** `b = 4` |

Đáp án `[2, 4]`. Bài toán chỉ là **đếm** rồi tìm hộp có `2` và hộp có `0`. Ngoài ra còn có thể suy ra bằng **tổng**: tổng thực `1 + 3 + 2 + 2 = 8` còn tổng đúng `1 + 2 + 3 + 4 = 10`; hiệu `8 - 10 = -2 = a - b`. Ta sẽ khai thác ý này ở các cách sau.

In [8]:
grid = [[1, 3], [2, 2]]
flat = [v for row in grid for v in row]
N = len(grid) ** 2
print("tổng thực:", sum(flat), "| tổng đúng:", N * (N + 1) // 2, "| a - b =", sum(flat) - N * (N + 1) // 2)
print("tổng bình phương thực:", sum(v * v for v in flat), "| đúng:", N * (N + 1) * (2 * N + 1) // 6)

tổng thực: 8 | tổng đúng: 10 | a - b = -2
tổng bình phương thực: 18 | đúng: 30


## 5. Cách 1 — Brute Force / baseline: mảng đếm tần suất

**Ý tưởng.** Trải phẳng lưới, đếm tần suất bằng `cnt[0..N]`, rồi duyệt `v` từ `1` đến `N`: `cnt[v] == 2` là `a`, `cnt[v] == 0` là `b`. Đây là mô phỏng trực tiếp định nghĩa.

In [9]:
def find_missing_and_repeated_count(grid: List[List[int]]) -> List[int]:
    n = len(grid)
    N = n * n
    cnt = [0] * (N + 1)
    for row in grid:
        for v in row:
            cnt[v] += 1
    a = b = -1
    for v in range(1, N + 1):
        if cnt[v] == 2:
            a = v                              # số lặp
        elif cnt[v] == 0:
            b = v                              # số thiếu
    return [a, b]


for grid, expected in examples:
    print(grid, "->", find_missing_and_repeated_count(grid), "(kỳ vọng", expected, ")")

[[1, 3], [2, 2]] -> [2, 4] (kỳ vọng [2, 4] )
[[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> [9, 5] (kỳ vọng [9, 5] )
[[1, 1], [3, 4]] -> [1, 2] (kỳ vọng [1, 2] )


In [10]:
# Xem mảng đếm của ví dụ 2
grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]
N = len(grid) ** 2
cnt = [0] * (N + 1)
for row in grid:
    for v in row:
        cnt[v] += 1
for v in range(1, N + 1):
    note = "  <- LẶP" if cnt[v] == 2 else "  <- THIẾU" if cnt[v] == 0 else ""
    print(f"cnt[{v}] = {cnt[v]}{note}")

cnt[1] = 1
cnt[2] = 1
cnt[3] = 1
cnt[4] = 1
cnt[5] = 0  <- THIẾU
cnt[6] = 1
cnt[7] = 1
cnt[8] = 1
cnt[9] = 2  <- LẶP


**Độ phức tạp (Cách 1).** Đọc `N = n²` ô rồi duyệt `N` giá trị: **Time `O(n²)`**; mảng đếm kích thước `N + 1`: **Space `O(n²)`**.

**Nút thắt.** Thời gian `O(n²)` là tối ưu (phải đọc mọi ô), nhưng bộ nhớ `O(n²)` hơi dư. Ta có thể giảm xuống `O(1)` nhờ toán học.

## 6. Ý tưởng thuật toán / bước đột phá

Vì "gần như" mọi giá trị `1..N` xuất hiện đúng một lần, **tổng** và **tổng bình phương** của lưới chỉ khác "bản chuẩn" ở hai giá trị `a` và `b`:

- `sum(grid) - N(N + 1)/2 = a - b` (ký hiệu `d`);
- `sum(grid²) - N(N + 1)(2N + 1)/6 = a² - b²` (ký hiệu `e`).

Vì `a² - b² = (a - b)(a + b)`, ta suy ra `a + b = e / d` (`d ≠ 0` vì `a ≠ b`). Giải hệ hai ẩn: `a = (d + e/d) / 2`, `b = a - d`. Chỉ cần vài biến số, **không cần mảng đếm**: `O(1)` bộ nhớ.

Một hướng khác: **XOR** của mọi giá trị trong lưới và mọi số `1..N` chỉ còn `a ^ b` (các số khác xuất hiện đúng hai lần nên triệt tiêu); tách theo một bit mà `a` và `b` khác nhau để tìm riêng từng số.

## 7. Các cách tiếp theo

### Cách 2 — `set` để tìm số lặp, hiệu tổng để tìm số thiếu

**Giải thích.** Duyệt lưới, dùng `set` `seen` để phát hiện số lặp `a` (gặp lại một giá trị đã có). Khi đó **tổng thực** `S` đã chứa `a` hai lần và thiếu `b`: `S = N(N + 1)/2 + a - b`, suy ra `b = N(N + 1)/2 + a - S`.

In [11]:
def find_missing_and_repeated_set(grid: List[List[int]]) -> List[int]:
    N = len(grid) ** 2
    seen = set()
    total = 0                                   # tổng thực của lưới
    a = -1
    for row in grid:
        for v in row:
            if v in seen:
                a = v                           # gặp lại -> số lặp
            seen.add(v)
            total += v
    b = N * (N + 1) // 2 + a - total            # tổng đúng + a - tổng thực
    return [a, b]


for grid, expected in examples:
    print(grid, "->", find_missing_and_repeated_set(grid), "(kỳ vọng", expected, ")")

[[1, 3], [2, 2]] -> [2, 4] (kỳ vọng [2, 4] )
[[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> [9, 5] (kỳ vọng [9, 5] )
[[1, 1], [3, 4]] -> [1, 2] (kỳ vọng [1, 2] )


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(n²)`** cho `set`.

### Cách 3 — Hai phương trình: tổng và tổng bình phương

**Giải thích.** Tính `d = tổng thực - N(N + 1)/2 = a - b` và `e = tổng bình phương thực - N(N + 1)(2N + 1)/6 = a² - b²`. Khi đó `a + b = e // d`, nên `a = (d + e // d) // 2` và `b = a - d`. Mọi phép chia ở đây đều **chia hết** (ta kiểm chứng ở phần test), nên dùng `//` là an toàn.

In [12]:
def find_missing_and_repeated_math(grid: List[List[int]]) -> List[int]:
    N = len(grid) ** 2
    s = sq = 0
    for row in grid:
        for v in row:
            s += v
            sq += v * v
    d = s - N * (N + 1) // 2                    # a - b
    e = sq - N * (N + 1) * (2 * N + 1) // 6     # a^2 - b^2 = (a - b)(a + b)
    plus = e // d                               # a + b
    a = (d + plus) // 2
    b = a - d
    return [a, b]


for grid, expected in examples:
    print(grid, "->", find_missing_and_repeated_math(grid), "(kỳ vọng", expected, ")")

[[1, 3], [2, 2]] -> [2, 4] (kỳ vọng [2, 4] )
[[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> [9, 5] (kỳ vọng [9, 5] )
[[1, 1], [3, 4]] -> [1, 2] (kỳ vọng [1, 2] )


In [13]:
# Từng đại lượng của ví dụ 2
grid = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]
N = len(grid) ** 2
flat = [v for row in grid for v in row]
d = sum(flat) - N * (N + 1) // 2
e = sum(v * v for v in flat) - N * (N + 1) * (2 * N + 1) // 6
print(f"d = a - b = {d}; e = a^2 - b^2 = {e}; a + b = e / d = {e // d}")
a = (d + e // d) // 2
print(f"a = (d + (a+b)) / 2 = {a}; b = a - d = {a - d}")

d = a - b = 4; e = a^2 - b^2 = 56; a + b = e / d = 14
a = (d + (a+b)) / 2 = 9; b = a - d = 5


**Độ phức tạp Cách 3:** **Time `O(n²)`**, **Space `O(1)`**. (Số lớn nhất: `sq <= 2500 · 2500²/... ~ 5·10⁹`, Python `int` không tràn.)

### Cách 4 — XOR và chia hai nhóm theo bit thấp nhất

**Giải thích.**

1. Đặt `x` = XOR của **mọi giá trị trong lưới** và **mọi số `1..N`**. Mọi số bình thường xuất hiện `1 + 1 = 2` lần (triệt tiêu); `a` xuất hiện `2 + 1 = 3` lần (còn lại `a`); `b` xuất hiện `0 + 1 = 1` lần (còn lại `b`). Nên `x = a ^ b`, và `x ≠ 0` vì `a ≠ b`.
2. Lấy `lb = x & -x`, **một bit mà `a` và `b` khác nhau**.
3. Chia mọi số (cả trong lưới lẫn `1..N`) thành hai nhóm theo việc có bit `lb` hay không. Trong mỗi nhóm, các số bình thường vẫn triệt tiêu và chỉ còn một trong `a`, `b`. Ta được hai số `p`, `q`.
4. Đếm `p` trong lưới: nếu `p` xuất hiện `2` lần thì `p = a`, ngược lại `p = b` (và `q` là số còn lại).

In [14]:
def find_missing_and_repeated_xor(grid: List[List[int]]) -> List[int]:
    N = len(grid) ** 2
    flat = [v for row in grid for v in row]
    x = 0
    for v in flat:
        x ^= v
    for v in range(1, N + 1):
        x ^= v                                  # x = a ^ b
    lb = x & -x                                 # một bit mà a và b khác nhau
    p = q = 0
    for v in flat + list(range(1, N + 1)):
        if v & lb:
            p ^= v
        else:
            q ^= v
    # p, q là {a, b} theo thứ tự nào đó: số nào xuất hiện trong lưới hai lần là a
    if flat.count(p) == 2:
        return [p, q]
    return [q, p]


for grid, expected in examples:
    print(grid, "->", find_missing_and_repeated_xor(grid), "(kỳ vọng", expected, ")")

# Quan sát ví dụ 1 bằng nhị phân
grid = [[1, 3], [2, 2]]
flat = [v for row in grid for v in row]
x = 0
for v in flat + [1, 2, 3, 4]:
    x ^= v
print(f"x = a ^ b = {x} ({bin(x)}), lb = {x & -x} ({bin(x & -x)})  (a=2=0b10, b=4=0b100)")

[[1, 3], [2, 2]] -> [2, 4] (kỳ vọng [2, 4] )
[[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> [9, 5] (kỳ vọng [9, 5] )
[[1, 1], [3, 4]] -> [1, 2] (kỳ vọng [1, 2] )
x = a ^ b = 6 (0b110), lb = 2 (0b10)  (a=2=0b10, b=4=0b100)


**Độ phức tạp Cách 4:** **Time `O(n²)`**, **Space `O(n²)`** vì dựng `flat` (có thể duyệt trực tiếp hai tầng để còn `O(1)`; `flat.count` là `O(n²)`).

In [15]:
# Code nộp LeetCode cho Cách 1 (mảng đếm: rõ ràng, dễ kiểm tra)
from typing import List

class Solution:
    def findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]:
        n = len(grid)
        N = n * n
        cnt = [0] * (N + 1)
        for row in grid:
            for v in row:
                cnt[v] += 1
        a = b = -1
        for v in range(1, N + 1):
            if cnt[v] == 2:
                a = v
            elif cnt[v] == 0:
                b = v
        return [a, b]

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 (hệ phương trình) trên Ví dụ 1 và (b) Cách 4 (XOR hai nhóm) trên Ví dụ 2.

In [16]:
def trace_math(grid):
    N = len(grid) ** 2
    flat = [v for row in grid for v in row]
    s, sq = 0, 0
    for v in flat:
        s += v
        sq += v * v
        print(f"v={v}: tổng={s:>3} tổng bình phương={sq:>4}")
    exp_s = N * (N + 1) // 2
    exp_sq = N * (N + 1) * (2 * N + 1) // 6
    d, e = s - exp_s, sq - exp_sq
    print(f"đúng: tổng={exp_s}, tổng bình phương={exp_sq}")
    print(f"d = a - b = {s} - {exp_s} = {d}")
    print(f"e = a^2 - b^2 = {sq} - {exp_sq} = {e}")
    plus = e // d
    a = (d + plus) // 2
    b = a - d
    print(f"a + b = e / d = {plus} -> a = (d + (a+b)) / 2 = {a}, b = a - d = {b}")
    return [a, b]


assert trace_math([[1, 3], [2, 2]]) == [2, 4]

v=1: tổng=  1 tổng bình phương=   1
v=3: tổng=  4 tổng bình phương=  10
v=2: tổng=  6 tổng bình phương=  14
v=2: tổng=  8 tổng bình phương=  18
đúng: tổng=10, tổng bình phương=30
d = a - b = 8 - 10 = -2
e = a^2 - b^2 = 18 - 30 = -12
a + b = e / d = 6 -> a = (d + (a+b)) / 2 = 2, b = a - d = 4


In [17]:
def trace_xor(grid):
    N = len(grid) ** 2
    flat = [v for row in grid for v in row]
    x = 0
    for v in flat + list(range(1, N + 1)):
        x ^= v
    lb = x & -x
    print(f"x = a ^ b = {x} = {bin(x)} | bit thấp nhất khác nhau lb = {lb} = {bin(lb)}")
    g1, g0 = [], []
    for v in flat + list(range(1, N + 1)):
        (g1 if v & lb else g0).append(v)
    p = q = 0
    for v in g1:
        p ^= v
    for v in g0:
        q ^= v
    print("nhóm có bit lb :", sorted(g1), "-> XOR =", p)
    print("nhóm không có  :", sorted(g0), "-> XOR =", q)
    a = p if flat.count(p) == 2 else q
    b = q if a == p else p
    print(f"{p} xuất hiện {flat.count(p)} lần trong lưới -> a = {a}, b = {b}")
    return [a, b]


assert trace_xor([[9, 1, 7], [8, 9, 2], [3, 4, 6]]) == [9, 5]

x = a ^ b = 12 = 0b1100 | bit thấp nhất khác nhau lb = 4 = 0b100
nhóm có bit lb : [4, 4, 5, 6, 6, 7, 7] -> XOR = 5
nhóm không có  : [1, 1, 2, 2, 3, 3, 8, 8, 9, 9, 9] -> XOR = 9
5 xuất hiện 0 lần trong lưới -> a = 9, b = 5


Bảng tóm tắt (Cách 3, Ví dụ 1: `[[1, 3], [2, 2]]`, `N = 4`):

| đại lượng | giá trị |
|---|---|
| tổng thực / đúng | `8` / `10` |
| `d = a - b` | `-2` |
| tổng bình phương thực / đúng | `18` / `30` |
| `e = a² - b²` | `-12` |
| `a + b = e / d` | `6` |
| `a = (d + 6) / 2`, `b = a - d` | `2`, `4` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (mảng đếm). Đúng class `Solution`, đúng method `findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]`.

In [18]:
from typing import List

class Solution:
    def findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]:
        n = len(grid)
        N = n * n                                   # các số cần có là 1..N
        cnt = [0] * (N + 1)                          # cnt[v] = số lần v xuất hiện (bỏ chỉ số 0)
        for row in grid:
            for v in row:
                cnt[v] += 1
        a = b = -1
        for v in range(1, N + 1):
            if cnt[v] == 2:
                a = v                               # số lặp
            elif cnt[v] == 0:
                b = v                               # số thiếu
        return [a, b]                               # [lặp, thiếu]


sol = Solution()
for grid, expected in examples:
    print(grid, "->", sol.findMissingAndRepeatedValues(grid))

[[1, 3], [2, 2]] -> [2, 4]
[[9, 1, 7], [8, 9, 2], [3, 4, 6]] -> [9, 5]
[[1, 1], [3, 4]] -> [1, 2]


## 10. Vì sao thuật toán đúng

- **Cách 1.** Theo đề, mỗi `v` trong `[1, N]` xuất hiện `1` lần, trừ `a` (`2` lần) và `b` (`0` lần). Mảng đếm ghi đúng số lần, nên đúng một `v` có `cnt[v] == 2` (chính là `a`) và đúng một `v` có `cnt[v] == 0` (chính là `b`). Đề đảm bảo tồn tại duy nhất nên không cần xét trường hợp khác.
- **Cách 2.** Tổng thực `S = (1 + ... + N) - b + a` (mất `b`, dư thêm một `a`), suy ra `b = N(N+1)/2 + a - S`. Phát hiện `a` bằng `set` là đúng vì chỉ `a` bị gặp lại.
- **Cách 3.** `d = a - b`, `e = a² - b²`; vì `a ≠ b` nên `d ≠ 0`, `a + b = e / d` chính xác, và `a = (d + (a + b)) / 2`, `b = a - d`.
- **Cách 4.** XOR triệt tiêu các số xuất hiện chẵn lần; kết quả `a ^ b ≠ 0` có bit `lb` mà đúng **một** trong hai số có. Hai nhóm tách theo `lb` mỗi nhóm chứa đúng một trong `a`, `b`, các số còn lại xuất hiện hai lần trong từng nhóm. Cuối cùng xác định số lặp bằng cách đếm trong lưới.
- **Thứ tự trả về** là `[a, b]` = `[lặp, thiếu]` theo đề.

## 11. Phân tích Time / Space Complexity

Gọi `n` là cạnh lưới, `N = n²` là số ô.

- **Cách 1 (mảng đếm):** Time `O(N)`, Space `O(N)`.
- **Cách 2 (`set` + hiệu tổng):** Time `O(N)`, Space `O(N)`.
- **Cách 3 (tổng + tổng bình phương):** Time `O(N)`, Space `O(1)`.
- **Cách 4 (XOR):** Time `O(N)`, Space `O(1)` nếu không dựng `flat` (hoặc `O(N)` như bản demo).
- Phải đọc mọi ô nên `O(N) = O(n²)` là tối ưu. Ở đây `n <= 50`, tức tối đa `2 500` ô.

## 12. Bộ test

In [19]:
sol = Solution()
impls = {
    "count": find_missing_and_repeated_count,
    "set": find_missing_and_repeated_set,
    "math": find_missing_and_repeated_math,
    "xor": find_missing_and_repeated_xor,
    "submit": sol.findMissingAndRepeatedValues,
}

for grid, expected in examples:
    for key, fn in impls.items():
        got = fn([r[:] for r in grid])
        assert got == expected, (key, grid, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(grid, expected, label):
    for key, fn in impls.items():
        got = fn([r[:] for r in grid])
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([[1, 1], [3, 4]], [1, 2], "n = 2: số lặp nhỏ nhất, số thiếu 2")
check_all([[4, 4], [2, 3]], [4, 1], "n = 2: thiếu số 1, lặp số lớn nhất N")
check_all([[1, 2], [2, 4]], [2, 3], "n = 2: lặp 2, thiếu 3")
check_all([[1, 2, 3], [4, 5, 6], [7, 8, 8]], [8, 9], "n = 3: thiếu số lớn nhất N = 9")
check_all([[2, 2, 3], [4, 5, 6], [7, 8, 9]], [2, 1], "n = 3: thiếu số nhỏ nhất 1")
check_all([[1, 2, 3], [4, 5, 6], [7, 9, 9]], [9, 8], "n = 3: hai số lặp nằm cạnh nhau")
check_all([[9, 8, 7], [6, 5, 4], [3, 2, 2]], [2, 1], "n = 3: thứ tự giảm dần")

# n = 50: lưới lớn, lặp 2500, thiếu 1
big = list(range(2, 2501)) + [2500]
check_all([big[i * 50:(i + 1) * 50] for i in range(50)], [2500, 1], "n = 50: lặp N = 2500, thiếu 1")
big2 = [1] + list(range(1, 2500))
check_all([big2[i * 50:(i + 1) * 50] for i in range(50)], [1, 2500], "n = 50: lặp 1, thiếu 2500")

PASS n = 2: số lặp nhỏ nhất, số thiếu 2 -> [1, 2]
PASS n = 2: thiếu số 1, lặp số lớn nhất N -> [4, 1]
PASS n = 2: lặp 2, thiếu 3 -> [2, 3]
PASS n = 3: thiếu số lớn nhất N = 9 -> [8, 9]
PASS n = 3: thiếu số nhỏ nhất 1 -> [2, 1]
PASS n = 3: hai số lặp nằm cạnh nhau -> [9, 8]
PASS n = 3: thứ tự giảm dần -> [2, 1]
PASS n = 50: lặp N = 2500, thiếu 1 -> [2500, 1]
PASS n = 50: lặp 1, thiếu 2500 -> [1, 2500]


### 12.2. Tính chất và phản ví dụ

In [21]:
def make_grid(n, a, b, rng):
    """Tạo lưới hợp lệ: hoán vị 1..N, thay b bằng a (a != b), xáo trộn."""
    N = n * n
    vals = list(range(1, N + 1))
    vals[b - 1] = a
    rng.shuffle(vals)
    return [vals[i * n:(i + 1) * n] for i in range(n)]


def is_valid_answer(grid, ans):
    N = len(grid) ** 2
    c = Counter(v for row in grid for v in row)
    a_true = [v for v in range(1, N + 1) if c[v] == 2]
    b_true = [v for v in range(1, N + 1) if c[v] == 0]
    return len(ans) == 2 and a_true == [ans[0]] and b_true == [ans[1]]


# Tính chất 1: hàm kiểm tra khớp định nghĩa, tổng/tổng bình phương
rng = random.Random(11)
for _ in range(300):
    n = rng.randint(2, 8)
    N = n * n
    a, b = rng.sample(range(1, N + 1), 2)
    g = make_grid(n, a, b, rng)
    assert sol.findMissingAndRepeatedValues(g) == [a, b]
    flat = [v for row in g for v in row]
    assert sum(flat) - N * (N + 1) // 2 == a - b
    assert sum(v * v for v in flat) - N * (N + 1) * (2 * N + 1) // 6 == a * a - b * b
    assert is_valid_answer(g, [a, b])
print("PASS tổng và tổng bình phương đúng như công thức d = a - b, e = a^2 - b^2")

# Tính chất 2: phép chia e // d luôn chia hết (an toàn dùng //)
for _ in range(300):
    n = rng.randint(2, 20)
    N = n * n
    a, b = rng.sample(range(1, N + 1), 2)
    d, e = a - b, a * a - b * b
    assert e % d == 0 and e // d == a + b
print("PASS e chia hết cho d")

# Tính chất 3: xáo trộn các ô không đổi đáp án
g = [[9, 1, 7], [8, 9, 2], [3, 4, 6]]
flat = [v for row in g for v in row]
for _ in range(50):
    rng.shuffle(flat)
    g2 = [flat[0:3], flat[3:6], flat[6:9]]
    assert sol.findMissingAndRepeatedValues(g2) == [9, 5]
print("PASS xáo trộn không đổi đáp án")

# Tính chất 4: đầu vào không bị sửa
copy = [r[:] for r in g]
for key, fn in impls.items():
    fn(g)
assert g == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: trả về sai thứ tự
def sai_thu_tu(grid):
    a, b = sol.findMissingAndRepeatedValues(grid)
    return [b, a]

assert sai_thu_tu([[1, 3], [2, 2]]) == [4, 2] != [2, 4]
print("PASS phản ví dụ: [thiếu, lặp] cho [4, 2] thay vì [2, 4]")

# Phản ví dụ 2: quên bù số lặp khi tính số thiếu
def sai_quen_bu(grid):
    N = len(grid) ** 2
    seen, total, a = set(), 0, -1
    for row in grid:
        for v in row:
            if v in seen:
                a = v
            seen.add(v)
            total += v
    return [a, N * (N + 1) // 2 - total]          # thiếu "+ a"

assert sai_quen_bu([[1, 3], [2, 2]]) == [2, 2] != [2, 4]
print("PASS phản ví dụ: b = tổng đúng - tổng thực cho", sai_quen_bu([[1, 3], [2, 2]]), "thay vì [2, 4]")

PASS tổng và tổng bình phương đúng như công thức d = a - b, e = a^2 - b^2
PASS e chia hết cho d
PASS xáo trộn không đổi đáp án
PASS đầu vào không bị sửa
PASS phản ví dụ: [thiếu, lặp] cho [4, 2] thay vì [2, 4]
PASS phản ví dụ: b = tổng đúng - tổng thực cho [2, 2] thay vì [2, 4]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
rng = random.Random(2965)
for trial in range(1500):
    n = rng.randint(2, 50)
    N = n * n
    a, b = rng.sample(range(1, N + 1), 2)
    g = make_grid(n, a, b, rng)
    for key, fn in impls.items():
        got = fn([r[:] for r in g])
        assert got == [a, b], (key, n, got, [a, b])
        assert is_valid_answer(g, got)
print("PASS 1500 bộ ngẫu nhiên (n tới 50) - mọi cách khớp nhau và khớp đáp án gốc")

PASS 1500 bộ ngẫu nhiên (n tới 50) - mọi cách khớp nhau và khớp đáp án gốc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về `[thiếu, lặp]` thay vì `[lặp, thiếu]`

In [23]:
def sai_thu_tu_2(grid):
    N = len(grid) ** 2
    cnt = [0] * (N + 1)
    for row in grid:
        for v in row:
            cnt[v] += 1
    a = next(v for v in range(1, N + 1) if cnt[v] == 2)
    b = next(v for v in range(1, N + 1) if cnt[v] == 0)
    return [b, a]                                # đảo ngược


print("sai  :", sai_thu_tu_2([[1, 3], [2, 2]]), "| đúng:", Solution().findMissingAndRepeatedValues([[1, 3], [2, 2]]))

sai  : [4, 2] | đúng: [2, 4]


**Vì sao sai:** đề quy định `ans[0] = a` (số **lặp**) và `ans[1] = b` (số **thiếu**). Đảo thứ tự cho `[4, 2]` thay vì `[2, 4]`. Luôn ghi chú trong code đâu là "lặp", đâu là "thiếu".

### Lỗi 2: Dùng `n` thay vì `n * n` làm cận trên

In [24]:
def sai_can_tren(grid):
    n = len(grid)
    cnt = [0] * (n + 1)                          # chỉ đủ chỗ cho giá trị <= n
    try:
        for row in grid:
            for v in row:
                cnt[v] += 1
    except IndexError as e:
        return "IndexError: " + str(e)
    return cnt


print("sai  :", sai_can_tren([[1, 3], [2, 2]]), "| đúng:", Solution().findMissingAndRepeatedValues([[1, 3], [2, 2]]))

sai  : IndexError: list index out of range | đúng: [2, 4]


**Vì sao sai:** các giá trị chạy tới **`n²`** (với `n = 2` là `4`), không phải `n`. Mảng đếm chỉ có `n + 1` ô nên chỉ số `3` đã ngoài biên. Hãy đặt `N = n * n` và dùng `[0] * (N + 1)`.

### Lỗi 3: Tính số thiếu mà quên bù số lặp

In [25]:
def sai_quen_bu_2(grid):
    N = len(grid) ** 2
    seen, total, a = set(), 0, -1
    for row in grid:
        for v in row:
            if v in seen:
                a = v
            seen.add(v)
            total += v
    b = N * (N + 1) // 2 - total                 # sai: tổng thực có dư a và thiếu b
    return [a, b]


print("sai  :", sai_quen_bu_2([[1, 3], [2, 2]]), "| đúng:", Solution().findMissingAndRepeatedValues([[1, 3], [2, 2]]))

sai  : [2, 2] | đúng: [2, 4]


**Vì sao sai:** tổng thực `= tổng đúng - b + a` (mất `b`, dư thêm `a`). Do đó `tổng đúng - tổng thực = b - a`, **không phải** `b`. Công thức đúng: `b = tổng đúng + a - tổng thực`.

### Lỗi 4: `return` ngay khi tìm thấy số lặp

In [26]:
def sai_return_som(grid):
    seen = set()
    for row in grid:
        for v in row:
            if v in seen:
                return [v]                       # mới tìm được a, chưa có b
            seen.add(v)
    return []


print("sai  :", sai_return_som([[1, 3], [2, 2]]), "| đúng:", Solution().findMissingAndRepeatedValues([[1, 3], [2, 2]]))

sai  : [2] | đúng: [2, 4]


**Vì sao sai:** đề cần **cả hai** số. Dừng ngay khi thấy số lặp thì chưa biết số thiếu, và còn có thể bỏ lỡ dữ liệu cần để tính số thiếu (ví dụ tổng của toàn bộ lưới). Phải đọc hết lưới hoặc ít nhất hết mọi phần cần để tìm `b`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "các số `1..N`, **một số lặp một số thiếu**" → ba công cụ kinh điển: **mảng đếm**, **tổng (và tổng bình phương)**, **XOR**.
- Công thức nhớ: `a - b = tổng thực - N(N+1)/2`; `a² - b² = tổng bp thực - N(N+1)(2N+1)/6`; `a + b = (a² - b²)/(a - b)`.
- Trải phẳng lưới: `[v for row in grid for v in row]` (đọc từ trái sang phải).
- Lưới `n x n` → số ô `N = n * n`, không phải `n`.
- Giá trị nhỏ (`<= 2500`) thì mảng đếm đơn giản, an toàn; bộ nhớ chặt hoặc `N` rất lớn thì dùng toán/XOR.
- Khi hai nghiệm cần thứ tự cụ thể (`[lặp, thiếu]`), viết ra giấy "vị trí 0 là ..., vị trí 1 là ..." trước khi `return`.

## 15. Biến thể của bài

1. Mảng một chiều `nums` độ dài `n` có các số `1..n`, một số lặp một số thiếu ("Set Mismatch", LeetCode 645).
2. Trường hợp **chỉ thiếu một số** (không lặp): tổng hiệu hoặc XOR là đủ.
3. **Hai số thiếu** và hai vị trí dư (tổng quát hóa hệ phương trình).
4. Tìm **mọi** số xuất hiện hai lần khi nhiều số có thể lặp ("Find All Duplicates in an Array").
5. Giải bằng **sắp xếp**: sau khi sắp xếp, so sánh phần tử với phần tử trước và với `i + 1`.

Thử biến thể 1, 2 và 5:

In [27]:
def set_mismatch(nums):
    n = len(nums)
    d = sum(nums) - n * (n + 1) // 2
    e = sum(v * v for v in nums) - n * (n + 1) * (2 * n + 1) // 6
    plus = e // d
    a = (d + plus) // 2
    return [a, a - d]


def one_missing(nums, n):
    """nums chứa n - 1 số khác nhau trong 1..n: tìm số thiếu bằng XOR."""
    x = 0
    for v in range(1, n + 1):
        x ^= v
    for v in nums:
        x ^= v
    return x


def by_sorting(grid):
    flat = sorted(v for row in grid for v in row)
    a = -1
    for i in range(1, len(flat)):
        if flat[i] == flat[i - 1]:                 # sau khi sắp xếp, số lặp đứng cạnh nhau
            a = flat[i]
    present = set(flat)
    b = next(v for v in range(1, len(flat) + 1) if v not in present)    # số nhỏ nhất của 1..N vắng mặt
    return [a, b]


print("1.", set_mismatch([1, 2, 2, 4]))                          # [2, 3]
print("2.", one_missing([1, 2, 4, 5], 5))                        # 3
print("5.", by_sorting([[9, 1, 7], [8, 9, 2], [3, 4, 6]]))       # [9, 5]
assert set_mismatch([1, 2, 2, 4]) == [2, 3]
assert one_missing([1, 2, 4, 5], 5) == 3
assert by_sorting([[9, 1, 7], [8, 9, 2], [3, 4, 6]]) == [9, 5]

1. [2, 3]
2. 3
5. [9, 5]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng đếm tần suất | `cnt[v] == 2` / `== 0` | `O(n²)` | `O(n²)` | **cách nộp bài**, dễ đúng, dễ debug |
| 2. `set` + hiệu tổng | `b = tổng đúng + a - tổng thực` | `O(n²)` | `O(n²)` | muốn bắt số lặp nhanh, tính thiếu bằng tổng |
| 3. Tổng + tổng bình phương | giải `a - b`, `a² - b²` | `O(n²)` | `O(1)` | bộ nhớ O(1), cần hiểu toán |
| 4. XOR chia hai nhóm | `a ^ b`, bit thấp nhất | `O(n²)` | `O(1)` | không tràn số trong ngôn ngữ cố định kiểu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 * (5 + 1) // 2`, `[v for row in [[1, 2], [3]] for v in row]`, `6 ^ 6 ^ 7`, `Solution().findMissingAndRepeatedValues([[2, 2], [3, 4]])`, `12 & -12`.

**Bài 2 (tính tay).** Với `grid = [[1, 5, 3], [4, 5, 6], [7, 8, 9]]`: lập bảng `cnt[1..9]`, tìm `a` và `b`; sau đó tính lại bằng `d = a - b` và `e = a² - b²`.

**Bài 3 (điền TODO).** Hoàn thành `find_todo(grid)` theo Cách 1.

In [28]:
def find_todo(grid):
    N = len(grid) ** 2
    cnt = [0] * (N + 1)
    for row in grid:
        for v in row:
            # TODO 1: tăng cnt[v] lên 1
            pass
    a = b = -1
    for v in range(1, N + 1):
        # TODO 2: nếu cnt[v] == 2 thì a = v; nếu cnt[v] == 0 thì b = v
        pass
    return [a, b]

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_math2(grid)` bằng **tổng và tổng bình phương** (Cách 3) mà không nhìn code mẫu. Kiểm tra với ba ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `find_two_missing(nums, n)`: `nums` chứa `n - 2` số khác nhau của `1..n`; trả về hai số thiếu (tăng dần), dùng tổng và tổng bình phương. (b) Viết `find_all_duplicates(nums)`: trả về mọi giá trị xuất hiện hai lần (mỗi giá trị tối đa hai lần). (c) Viết `find_with_xor_only(grid)` không dùng `list.count`: để xác định số nào là số lặp, hãy duyệt lưới một lần nữa bằng biến đếm.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `15`, `[1, 2, 3]`, `7`, `[2, 1]`, `4`. Bài 3: `cnt[v] += 1`; rồi `if cnt[v] == 2: a = v` và `elif cnt[v] == 0: b = v`.
- **Mức 2 (vừa):** Bài 2: các số có mặt `1, 5, 3, 4, 5, 6, 7, 8, 9`: `5` xuất hiện `2` lần, `2` xuất hiện `0` lần ⇒ `[5, 2]`. Kiểm tra: tổng thực `48`, tổng đúng `45` nên `d = 3 = 5 - 2`. Bài 4: `d = s - N(N+1)//2`, `e = sq - ...`, `a = (d + e // d) // 2`, `b = a - d`.
- **Mức 3 (sâu):** Bài 5(a): hai thiếu `p, q`: `p + q = tổng đúng - tổng thực` (`s`), `p² + q² = tổng bp đúng - tổng bp thực` (`t`); `p·q = (s² - t) / 2`; nghiệm từ `(p - q)² = 2t - s²`, rồi `p, q = (s ± r) / 2` với `r = sqrt(2t - s²)` (dùng `math.isqrt`). 5(b): đếm bằng `Counter` hoặc duyệt `seen`. 5(c): sau khi có `p`, đếm số lần `p` trong lưới bằng một vòng lặp, nếu `2` thì `p` là `a`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
from math import isqrt

# Bài 1
assert 5 * (5 + 1) // 2 == 15 and [v for row in [[1, 2], [3]] for v in row] == [1, 2, 3]
assert (6 ^ 6 ^ 7) == 7 and (12 & -12) == 4
assert Solution().findMissingAndRepeatedValues([[2, 2], [3, 4]]) == [2, 1]

# Bài 2
g2 = [[1, 5, 3], [4, 5, 6], [7, 8, 9]]
assert Solution().findMissingAndRepeatedValues(g2) == [5, 2]
flat2 = [v for row in g2 for v in row]
assert sum(flat2) - 45 == 3 == 5 - 2
assert sum(v * v for v in flat2) - 285 == 5 * 5 - 2 * 2

# Bài 3
def find_todo_ans(grid):
    N = len(grid) ** 2
    cnt = [0] * (N + 1)
    for row in grid:
        for v in row:
            cnt[v] += 1
    a = b = -1
    for v in range(1, N + 1):
        if cnt[v] == 2:
            a = v
        elif cnt[v] == 0:
            b = v
    return [a, b]

for grid, expected in examples:
    assert find_todo_ans(grid) == expected

# Bài 4
def find_math2(grid):
    N = len(grid) ** 2
    flat = [v for row in grid for v in row]
    d = sum(flat) - N * (N + 1) // 2
    e = sum(v * v for v in flat) - N * (N + 1) * (2 * N + 1) // 6
    a = (d + e // d) // 2
    return [a, a - d]

for grid, expected in examples:
    assert find_math2(grid) == expected

# Bài 5(a)
def find_two_missing(nums, n):
    s = n * (n + 1) // 2 - sum(nums)                                  # p + q
    t = n * (n + 1) * (2 * n + 1) // 6 - sum(v * v for v in nums)     # p^2 + q^2
    r = isqrt(2 * t - s * s)                                          # |p - q|
    return [(s - r) // 2, (s + r) // 2]

assert find_two_missing([1, 2, 5], 5) == [3, 4]
assert find_two_missing([2, 3, 4], 5) == [1, 5]
assert find_two_missing([1], 3) == [2, 3]

# Bài 5(b)
def find_all_duplicates(nums):
    seen, dup = set(), []
    for v in nums:
        if v in seen:
            dup.append(v)
        seen.add(v)
    return sorted(dup)

assert find_all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [2, 3]
assert find_all_duplicates([1, 2, 3]) == []

# Bài 5(c)
def find_with_xor_only(grid):
    N = len(grid) ** 2
    x = 0
    for row in grid:
        for v in row:
            x ^= v
    for v in range(1, N + 1):
        x ^= v
    lb = x & -x
    p = q = 0
    for row in grid:
        for v in row:
            if v & lb:
                p ^= v
            else:
                q ^= v
    for v in range(1, N + 1):
        if v & lb:
            p ^= v
        else:
            q ^= v
    times = 0
    for row in grid:
        for v in row:
            if v == p:
                times += 1
    return [p, q] if times == 2 else [q, p]

for grid, expected in examples:
    assert find_with_xor_only(grid) == expected
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng đếm tần suất; thứ tự trả về `[lặp, thiếu]`)

```python
from typing import List

class Solution:
    def findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]:
        N = len(grid) ** 2
        cnt = [0] * (N + 1)
        for row in grid:
            for v in row:
                cnt[v] += 1
        a = b = -1
        for v in range(1, N + 1):
            if cnt[v] == 2:
                a = v
            elif cnt[v] == 0:
                b = v
        return [a, b]
```

**Checklist**

- [ ] Cận trên là `N = n * n` (không phải `n`)?
- [ ] Mảng đếm có `N + 1` phần tử, truy cập `cnt[v]` với `v` trong `1..N`?
- [ ] Trả về đúng thứ tự `[lặp, thiếu]`?
- [ ] Nếu dùng tổng: `b = tổng đúng + a - tổng thực` (đã bù số lặp)?
- [ ] Nếu dùng `a² - b²`: đã chia cho `a - b` (khác `0`) và dùng `//` cho kết quả nguyên?
- [ ] Đã đọc hết lưới trước khi `return`?
- [ ] Đã thử: `n = 2`, số thiếu là `1`, số thiếu là `N`, số lặp là `N`, lưới `50 x 50`?
- [ ] Skeleton đúng `class Solution` / `def findMissingAndRepeatedValues(self, grid: List[List[int]]) -> List[int]`?